# LeetCode 2011 — Final Value of Variable After Performing Operations bằng Python

**Link đề:** https://leetcode.com/problems/final-value-of-variable-after-performing-operations/
**Độ khó:** Easy · **Chủ đề:** Array, String, Simulation

```python
class Solution:
    def finalValueAfterOperations(self, operations: List[str]) -> int:

```

## Mục tiêu học
1. Hiểu bài **mô phỏng (simulation)**: có trạng thái `X`, mỗi thao tác cập nhật trạng thái, trả về trạng thái cuối.
2. Nắm công cụ: so sánh chuỗi `==`, toán tử `in`, chỉ số chuỗi `op[1]`, `dict` ánh xạ, `+=`/`-=`, `sum` với generator, `str.count`.
3. Viết lời giải trực giác bằng `if/elif` liệt kê đủ 4 trường hợp, rồi rút gọn nhờ quan sát **ký tự giữa**.
4. Hiểu khác biệt **tiền tố `++X`** và **hậu tố `X++`** trong C/C++/Java — và vì sao ở bài này chúng như nhau.
5. So sánh 5 cách, trace, test, làm bài tập mô phỏng mở rộng (thêm phép toán, nhiều biến).

## 1. Đề bài yêu cầu gì?

Có một ngôn ngữ lập trình chỉ có **một biến `X`** và **bốn thao tác**:
- `++X` và `X++`: tăng `X` thêm 1.
- `--X` và `X--`: giảm `X` đi 1.

Ban đầu `X = 0`. Cho mảng chuỗi `operations`, trả về giá trị cuối của `X` sau khi thực hiện tất cả thao tác (theo thứ tự).

| `operations` | Kết quả | Quá trình |
|---|:-:|---|
| `["--X","X++","X++"]` | `1` | 0 → −1 → 0 → 1 |
| `["++X","++X","X++"]` | `3` | 0 → 1 → 2 → 3 |
| `["X++","++X","--X","X--"]` | `0` | 0 → 1 → 2 → 1 → 0 |

**Constraints:** `1 <= operations.length <= 100`; mỗi `operations[i]` là **đúng một trong 4 chuỗi** `"++X"`, `"X++"`, `"--X"`, `"X--"`.

In [1]:
operations = ["--X", "X++", "X++"]
X = 0
print("ban đầu X =", X)
for op in operations:
    if op in ("++X", "X++"):
        X += 1
    else:
        X -= 1
    print(f"{op}: X = {X}")

ban đầu X = 0
--X: X = -1
X++: X = 0
X++: X = 1


## 2. Nền tảng Python cần biết

### 2.1. Chuỗi và so sánh `==`

Hai chuỗi bằng nhau khi **mọi ký tự** giống nhau theo đúng thứ tự. `"++X" == "X++"` là `False` dù cùng chứa các ký tự giống nhau.

In [2]:
print("++X" == "++X")
print("++X" == "X++")
print(sorted("++X") == sorted("X++"))   # cùng tập ký tự, khác thứ tự

True
False
True


### 2.2. Chỉ số trong chuỗi

Mỗi chuỗi thao tác dài **3 ký tự**. `op[0]`, `op[1]`, `op[2]` là ký tự thứ nhất, hai, ba. Ký tự **giữa** `op[1]` của cả 4 thao tác:

| op | op[0] | op[1] | op[2] |
|---|:-:|:-:|:-:|
| `++X` | + | **+** | X |
| `X++` | X | **+** | + |
| `--X` | - | **-** | X |
| `X--` | X | **-** | - |

→ `op[1]` luôn là dấu của phép toán! Đây là quan sát then chốt cho cách gọn.

In [3]:
for op in ["++X", "X++", "--X", "X--"]:
    print(op, "->", "op[0] =", repr(op[0]), "| op[1] =", repr(op[1]), "| op[2] =", repr(op[2]))

++X -> op[0] = '+' | op[1] = '+' | op[2] = 'X'
X++ -> op[0] = 'X' | op[1] = '+' | op[2] = '+'
--X -> op[0] = '-' | op[1] = '-' | op[2] = 'X'
X-- -> op[0] = 'X' | op[1] = '-' | op[2] = '-'


### 2.3. Toán tử `in`

- Với chuỗi: `"+" in op` kiểm tra ký tự/chuỗi con `"+"` có nằm trong `op` không.
- Với tuple/list: `op in ("++X", "X++")` kiểm tra `op` có bằng **một trong** các phần tử không.

In [4]:
print("+" in "X++", "+" in "--X")
print("X++" in ("++X", "X++"), "--X" in ("++X", "X++"))

True False
True False


### 2.4. `+=` và `-=`

`X += 1` là viết tắt của `X = X + 1`. **Python không có** toán tử `++`/`--` như C/Java!

Thú vị: `++X` trong Python là **hợp lệ về cú pháp** nhưng nghĩa là `+(+X)` — dấu cộng một ngôi hai lần — **không thay đổi** `X`.

In [5]:
X = 5
X += 1
print("sau X += 1:", X)
X -= 1
print("sau X -= 1:", X)

X = 5
++X            # chỉ là +(+X), không tăng gì cả
print("sau dòng '++X' trong Python:", X)
print("--X trong Python =", --X, "(là -(-X))")

sau X += 1: 6
sau X -= 1: 5
sau dòng '++X' trong Python: 5
--X trong Python = 5 (là -(-X))


### 2.5. `dict` ánh xạ chuỗi → số

`dict` có thể dùng làm **bảng tra**: key là chuỗi thao tác, value là lượng thay đổi (`+1` hoặc `-1`).

In [6]:
delta = {"++X": 1, "X++": 1, "--X": -1, "X--": -1}
print(delta["X++"], delta["--X"])
print(len(delta), "thao tác")

1 -1
4 thao tác


### 2.6. Tiền tố vs hậu tố trong C/C++/Java (kiến thức mở rộng)

Trong C/Java:
- `++X` (**pre-increment**): tăng `X` trước, **biểu thức có giá trị mới**.
- `X++` (**post-increment**): biểu thức có **giá trị cũ**, rồi mới tăng.

```c
int X = 5;
int a = ++X;   // X = 6, a = 6
int b = X++;   // b = 6, X = 7
```

Khác biệt chỉ nằm ở **giá trị của biểu thức**. Bài này chỉ hỏi **giá trị cuối của X** (không dùng giá trị biểu thức) → `++X` và `X++` có tác dụng **y hệt** lên `X`. Ta mô phỏng khác biệt đó trong Python:

In [7]:
def pre_increment(state):      # ++X: tăng rồi trả giá trị mới
    state["X"] += 1
    return state["X"]

def post_increment(state):     # X++: trả giá trị cũ rồi tăng
    old = state["X"]
    state["X"] += 1
    return old

s = {"X": 5}
a = pre_increment(s)
print("a = ++X ->", a, "| X =", s["X"])
b_ = post_increment(s)
print("b = X++ ->", b_, "| X =", s["X"])

a = ++X -> 6 | X = 6
b = X++ -> 6 | X = 7


## 3. Cách 1 — Trực giác: `if/elif` liệt kê đủ 4 thao tác

Cách nghĩ đầu tiên: so sánh `op` với từng chuỗi một.

In [8]:
def final_value_if(operations):
    X = 0
    for op in operations:
        if op == "++X":
            X += 1
        elif op == "X++":
            X += 1
        elif op == "--X":
            X -= 1
        elif op == "X--":
            X -= 1
    return X

print(final_value_if(["--X", "X++", "X++"]))         # 1
print(final_value_if(["++X", "++X", "X++"]))         # 3
print(final_value_if(["X++", "++X", "--X", "X--"]))  # 0

1
3
0


### Code nộp LeetCode — Cách 1

In [9]:
from typing import List

class SolutionIf:
    def finalValueAfterOperations(self, operations: List[str]) -> int:
        X = 0
        for op in operations:
            if op == "++X" or op == "X++":
                X += 1
            else:
                X -= 1
        return X

print(SolutionIf().finalValueAfterOperations(["--X", "X++", "X++"]))

1


### Nút thắt (về độ gọn, không phải tốc độ)

Cách 1 đã `O(n)` — không thể nhanh hơn vì phải đọc mọi thao tác. "Nút thắt" ở đây là **độ rườm rà**: bốn nhánh, dễ gõ nhầm chuỗi (`"X+ +"`, `"+X+"`...). Câu hỏi: có đặc điểm **chung** nào phân biệt "tăng" và "giảm" bằng một phép kiểm tra duy nhất?

Nhìn bảng ở mục 2.2: **ký tự giữa** `op[1]` là `'+'` khi tăng, `'-'` khi giảm. Hoặc đơn giản hơn: "có dấu `+` trong chuỗi không?"

## 4. Cách 2 — Nhìn ký tự giữa `op[1]` (lời giải chính)

Giảng từng đoạn:

```python
X = 0                         # (1) trạng thái ban đầu
for op in operations:         # (2) thực hiện lần lượt
    if op[1] == '+':          # (3) ký tự giữa quyết định tăng/giảm
        X += 1
    else:
        X -= 1
return X                      # (4) trạng thái cuối
```

Vì đề **đảm bảo** chỉ có 4 chuỗi hợp lệ, `op[1]` chắc chắn là `'+'` hoặc `'-'` → nhánh `else` an toàn.

In [10]:
def final_value_middle(operations):
    X = 0
    for op in operations:
        if op[1] == '+':
            X += 1
        else:
            X -= 1
    return X

print(final_value_middle(["--X", "X++", "X++"]))
print(final_value_middle(["++X", "++X", "X++"]))
print(final_value_middle(["X++", "++X", "--X", "X--"]))

1
3
0


## 5. TRACE từng bước

Ví dụ 3: `["X++", "++X", "--X", "X--"]`.

| Bước | op | op[1] | Hành động | X trước | X sau |
|:-:|---|:-:|---|:-:|:-:|
| 1 | `X++` | + | tăng | 0 | 1 |
| 2 | `++X` | + | tăng | 1 | 2 |
| 3 | `--X` | − | giảm | 2 | 1 |
| 4 | `X--` | − | giảm | 1 | 0 |

Kết quả: **0**.

In [11]:
def trace_final_value(operations):
    X = 0
    print(f"operations = {operations}\nban đầu X = {X}")
    for step, op in enumerate(operations, start=1):
        before = X
        if op[1] == '+':
            X += 1
            action = "tăng"
        else:
            X -= 1
            action = "giảm"
        print(f"  Bước {step}: op = {op!r}, op[1] = {op[1]!r} -> {action}: {before} -> {X}")
    print("Kết quả:", X)
    return X

trace_final_value(["X++", "++X", "--X", "X--"])
print()
trace_final_value(["--X", "X++", "X++"])

operations = ['X++', '++X', '--X', 'X--']
ban đầu X = 0
  Bước 1: op = 'X++', op[1] = '+' -> tăng: 0 -> 1
  Bước 2: op = '++X', op[1] = '+' -> tăng: 1 -> 2
  Bước 3: op = '--X', op[1] = '-' -> giảm: 2 -> 1
  Bước 4: op = 'X--', op[1] = '-' -> giảm: 1 -> 0
Kết quả: 0

operations = ['--X', 'X++', 'X++']
ban đầu X = 0
  Bước 1: op = '--X', op[1] = '-' -> giảm: 0 -> -1
  Bước 2: op = 'X++', op[1] = '+' -> tăng: -1 -> 0
  Bước 3: op = 'X++', op[1] = '+' -> tăng: 0 -> 1
Kết quả: 1


1

`enumerate(operations, start=1)` đánh số bắt đầu từ **1** thay vì 0 — tiện cho việc in "Bước 1, 2, ...".

Một nhánh đáng chú ý: X có thể **âm** giữa chừng (ví dụ 1 đi qua −1). Không có ràng buộc nào cấm X âm.

In [12]:
trace_final_value(["--X", "X--", "--X"])

operations = ['--X', 'X--', '--X']
ban đầu X = 0
  Bước 1: op = '--X', op[1] = '-' -> giảm: 0 -> -1
  Bước 2: op = 'X--', op[1] = '-' -> giảm: -1 -> -2
  Bước 3: op = '--X', op[1] = '-' -> giảm: -2 -> -3
Kết quả: -3


-3

## 6. Code nộp LeetCode

In [13]:
class Solution:
    def finalValueAfterOperations(self, operations: List[str]) -> int:
        X = 0
        for op in operations:
            if op[1] == '+':
                X += 1
            else:
                X -= 1
        return X

for ops, expected in [(["--X", "X++", "X++"], 1), (["++X", "++X", "X++"], 3), (["X++", "++X", "--X", "X--"], 0)]:
    got = Solution().finalValueAfterOperations(ops)
    print(ops, "->", got)
    assert got == expected

['--X', 'X++', 'X++'] -> 1
['++X', '++X', 'X++'] -> 3
['X++', '++X', '--X', 'X--'] -> 0


## 7. Cách 3 — Bảng tra `dict`

Cách này **tách dữ liệu khỏi logic**: muốn thêm thao tác mới (ví dụ `"X+=2"`) chỉ cần thêm một dòng vào bảng, không sửa vòng lặp. Đây là tư duy rất hữu ích cho các bài mô phỏng lớn hơn.

In [14]:
DELTA = {"++X": 1, "X++": 1, "--X": -1, "X--": -1}

def final_value_dict(operations):
    X = 0
    for op in operations:
        X += DELTA[op]
    return X

print(final_value_dict(["--X", "X++", "X++"]))
print(final_value_dict(["X++", "++X", "--X", "X--"]))

1
0


## 8. Cách 4 — Một dòng với `sum` và generator expression

Giá trị cuối = (số thao tác tăng) − (số thao tác giảm) = tổng các `+1`/`-1`.

```python
sum(1 if op[1] == '+' else -1 for op in operations)
```

Biểu thức `A if điều_kiện else B` là **toán tử điều kiện** (ternary): trả `A` nếu điều kiện đúng, ngược lại `B`.

In [15]:
x = 7
print("chẵn" if x % 2 == 0 else "lẻ")

def final_value_sum(operations):
    return sum(1 if op[1] == '+' else -1 for op in operations)

print(final_value_sum(["--X", "X++", "X++"]))
print(final_value_sum(["++X", "++X", "X++"]))

lẻ
1
3


## 9. Cách 5 — Đếm: `tăng − giảm`

Gọi `p` = số thao tác chứa `'+'`, `n` = tổng số thao tác. Thì số thao tác giảm là `n − p`, và

$$X = p - (n - p) = 2p - n$$

Còn một mẹo nữa: ghép tất cả thao tác thành một chuỗi rồi đếm ký tự `'+'`: mỗi thao tác tăng có **đúng 2** dấu `+` → `p = count('+') // 2`.

In [16]:
def final_value_count(operations):
    p = sum(1 for op in operations if '+' in op)
    return 2 * p - len(operations)

def final_value_join(operations):
    s = "".join(operations)
    return (s.count('+') - s.count('-')) // 2

ops = ["X++", "++X", "--X", "X--", "X++"]
print("".join(ops))
print(final_value_count(ops), final_value_join(ops))

X++++X--XX--X++
1 1


`str.count(sub)` đếm số lần chuỗi con `sub` xuất hiện (không chồng nhau). `"".join(list_of_str)` nối các chuỗi lại với chuỗi rỗng ở giữa.

## 10. Vì sao đúng?

**Invariant (Cách 1, 2, 3):** sau khi xử lý `t` thao tác đầu, `X` = giá trị của biến sau đúng `t` thao tác đó. Khởi tạo `X = 0` (đúng đề); mỗi bước cộng đúng `+1`/`−1` theo loại thao tác → invariant duy trì; hết vòng `t = n` → `X` là giá trị cuối.

**Tính đúng của "ký tự giữa":** ta đã kiểm tra cả **4** chuỗi hợp lệ (bảng mục 2.2) — `op[1]` là `'+'` đúng với 2 chuỗi tăng và `'-'` đúng với 2 chuỗi giảm. Đề đảm bảo không có chuỗi nào khác.

**Cách 4, 5:** phép cộng có tính **giao hoán** → thứ tự thao tác không ảnh hưởng giá trị cuối; chỉ cần biết **bao nhiêu** lần tăng/giảm.

In [17]:
import itertools
ops = ["X++", "--X", "X++", "++X"]
results = {final_value_middle(list(p)) for p in itertools.permutations(ops)}
print("mọi hoán vị của", ops, "cho cùng kết quả:", results)

mọi hoán vị của ['X++', '--X', 'X++', '++X'] cho cùng kết quả: {2}


## 11. Time / Space và bảng so sánh

Gọi `n = len(operations)`. Mỗi chuỗi dài 3 (hằng số).

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. `if/elif` 4 nhánh | so sánh cả chuỗi | `O(n)` | `O(1)` | Trực giác, dài |
| 2. Ký tự giữa `op[1]` | `'+'` → tăng | `O(n)` | `O(1)` | **Khuyến nghị** |
| 3. Bảng `dict` | `X += DELTA[op]` | `O(n)` | `O(1)` (bảng 4 phần tử) | Dễ mở rộng |
| 4. `sum` + generator | tổng ±1 | `O(n)` | `O(1)` | Một dòng |
| 5. Đếm / `join` + `count` | `2p − n` | `O(n)` | `O(1)` / `O(n)` với `join` | Mẹo toán học |

Tất cả đều `O(n)` — tối ưu, vì phải đọc từng thao tác ít nhất một lần.

## 12. Bộ test: mẫu đề, biên, đối chiếu

1. Ba ví dụ đề.
2. Một thao tác (mỗi loại trong 4).
3. Toàn tăng (100 lần) → 100; toàn giảm → −100.
4. X âm giữa chừng.
5. Đối chiếu ngẫu nhiên giữa 6 cách (5 cách + `Solution`).

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<24} PASS ({actual})")

f = final_value_middle
check("đề 1", f(["--X", "X++", "X++"]), 1)
check("đề 2", f(["++X", "++X", "X++"]), 3)
check("đề 3", f(["X++", "++X", "--X", "X--"]), 0)
for op, exp in [("++X", 1), ("X++", 1), ("--X", -1), ("X--", -1)]:
    check(f"một thao tác {op}", f([op]), exp)
check("100 lần tăng", f(["X++"] * 100), 100)
check("100 lần giảm", f(["--X"] * 100), -100)

đề 1                     PASS (1)
đề 2                     PASS (3)
đề 3                     PASS (0)
một thao tác ++X         PASS (1)
một thao tác X++         PASS (1)
một thao tác --X         PASS (-1)
một thao tác X--         PASS (-1)
100 lần tăng             PASS (100)
100 lần giảm             PASS (-100)


`["X++"] * 100` tạo list gồm 100 phần tử `"X++"` — phép nhân list với số nguyên là **lặp lại** list.

In [19]:
import random

rng = random.Random(2011)
ALL = ["++X", "X++", "--X", "X--"]
methods = [final_value_if, final_value_middle, final_value_dict, final_value_sum,
           final_value_count, final_value_join, Solution().finalValueAfterOperations]
for _ in range(2000):
    ops = [rng.choice(ALL) for _ in range(rng.randint(1, 100))]
    answers = {m(ops) for m in methods}
    assert len(answers) == 1, (ops, answers)
print("2000 test ngẫu nhiên: 7 hàm cho cùng kết quả")

2000 test ngẫu nhiên: 7 hàm cho cùng kết quả


## 13. Lỗi người mới hay gặp

### Lỗi 1: Nhìn ký tự **đầu** `op[0]`

In [20]:
def wrong_first_char(operations):
    X = 0
    for op in operations:
        if op[0] == '+':
            X += 1
        else:
            X -= 1
    return X

print("sai :", wrong_first_char(["X++", "X++"]))   # -2
print("đúng:", final_value_middle(["X++", "X++"]))  # 2

sai : -2
đúng: 2


**Vì sao sai:** với hậu tố `X++`, `op[0]` là `'X'` → rơi vào nhánh `else` và bị **giảm**. **Sửa:** dùng `op[1]` (luôn là dấu), hoặc `'+' in op`.

### Lỗi 2: Chỉ xử lý dạng tiền tố

In [21]:
def wrong_prefix_only(operations):
    X = 0
    for op in operations:
        if op == "++X":
            X += 1
        elif op == "--X":
            X -= 1
    return X

print("sai :", wrong_prefix_only(["X++", "++X", "X--"]))   # 1
print("đúng:", final_value_middle(["X++", "++X", "X--"]))  # 1 — trùng hợp!
print("sai :", wrong_prefix_only(["X++", "X++"]))           # 0
print("đúng:", final_value_middle(["X++", "X++"]))          # 2

sai : 1
đúng: 1
sai : 0
đúng: 2


**Vì sao sai:** `X++` và `X--` bị **bỏ qua**. Ví dụ đầu trùng hợp ra đúng (tăng bị bỏ và giảm bị bỏ triệt tiêu) — bài học: **một test pass chưa chứng minh code đúng**. **Sửa:** xử lý đủ 4 thao tác.

### Lỗi 3: Viết `X++` hoặc `++X` trong Python

In [22]:
X = 0
for op in ["++X", "++X"]:
    ++X            # Python: +(+X) — không đổi X!
print("X sau 'tăng' hai lần kiểu C:", X)

code = "X = 0\nX++"
try:
    exec(code)
except SyntaxError as e:
    print("X++ trong Python -> SyntaxError:", e.msg)

X sau 'tăng' hai lần kiểu C: 0
X++ trong Python -> SyntaxError: invalid syntax


**Vì sao sai:** Python không có `++`/`--`. `++X` lặng lẽ không làm gì (lỗi khó phát hiện!), `X++` là lỗi cú pháp. **Sửa:** `X += 1`, `X -= 1`.

(`exec(code)` chạy một chuỗi như code Python — chỉ dùng ở đây để minh họa lỗi cú pháp mà không làm hỏng cả cell.)

### Lỗi 4: Đếm `'+'` mà quên chia 2 (Cách 5)

In [23]:
def wrong_count_plus(operations):
    s = "".join(operations)
    return s.count('+') - s.count('-')

print("sai :", wrong_count_plus(["X++", "++X", "--X"]))   # 2
print("đúng:", final_value_join(["X++", "++X", "--X"]))   # 1

sai : 2
đúng: 1


**Vì sao sai:** mỗi thao tác có **hai** dấu. **Sửa:** chia 2, hoặc đếm theo thao tác (`'+' in op`).

## 14. Mẹo nhận dạng pattern

- Đề "có trạng thái ban đầu, áp lần lượt các thao tác, hỏi trạng thái cuối" → **Simulation**: một biến trạng thái + vòng `for` + cập nhật.
- Tập thao tác hữu hạn, nhỏ → tìm **đặc điểm phân biệt tối thiểu** (ở đây là `op[1]`) hoặc dùng **bảng tra `dict`**.
- Thao tác chỉ cộng/trừ (giao hoán) → có thể **đếm** thay vì mô phỏng tuần tự.
- Khi đọc đề có cú pháp C/Java (`++X`, `X++`) — nhớ Python **không** hỗ trợ.

## 15. Biến thể

1. Thêm thao tác `"X*=2"` và `"X+=5"` → thứ tự giờ **quan trọng** (phép nhân không giao hoán với cộng), không đếm được nữa, phải mô phỏng.
2. Nhiều biến: `"++A"`, `"B--"` → dùng `dict` lưu nhiều biến.
3. **LeetCode 682 — Baseball Game:** mô phỏng với stack.
4. **LeetCode 1598 — Crawler Log Folder:** mô phỏng độ sâu thư mục.

In [24]:
def simulate_extended(operations):
    X = 0
    for op in operations:
        if op in ("++X", "X++"):
            X += 1
        elif op in ("--X", "X--"):
            X -= 1
        elif op == "X*=2":
            X *= 2
        elif op == "X+=5":
            X += 5
    return X

print(simulate_extended(["X++", "X*=2", "X+=5"]))   # (0+1)*2+5 = 7
print(simulate_extended(["X*=2", "X++", "X+=5"]))   # 0*2+1+5 = 6 — thứ tự quan trọng!

7
6


## 16. Bài tập tự luyện (TODO — chưa chạy)

### Bài 1 — Tính tay
Trace `["++X", "X--", "X--", "--X", "X++"]`: ghi `X` sau từng bước.

### Bài 2 — Dự đoán output
`wrong_first_char(["++X", "X--", "X++"])` trả về gì? So với đáp án đúng?

### Bài 3 — Tự viết lời giải bằng `op[1]`

In [25]:
def exercise_final_value(operations: list[str]) -> int:
    # TODO: X = 0; duyệt op; op[1] == '+' thì tăng, ngược lại giảm
    pass

# assert exercise_final_value(["--X", "X++", "X++"]) == 1
# assert exercise_final_value(["X++", "++X", "--X", "X--"]) == 0

### Bài 4 — Viết bằng công thức `2p − n` (không mô phỏng)

In [26]:
def exercise_final_value_formula(operations: list[str]) -> int:
    # TODO: p = số thao tác có '+'; return 2 * p - len(operations)
    pass

# assert exercise_final_value_formula(["++X", "++X", "X++"]) == 3

### Bài 5 — Biến thể nhiều biến: mỗi thao tác dạng `"++A"`, `"B--"`, ... (tên biến là một chữ cái in hoa). Trả về `dict` giá trị cuối các biến đã xuất hiện.

In [27]:
def exercise_multi_vars(operations: list[str]) -> dict:
    # TODO: tên biến là ký tự không phải '+'/'-'; dấu là op[1]
    pass

# assert exercise_multi_vars(["++A", "B--", "A++", "--B", "C++"]) == {"A": 2, "B": -2, "C": 1}

### Bài 6 — Mini challenge: LeetCode 682 — Baseball Game
Mỗi thao tác là số nguyên (ghi điểm), `"+"` (điểm = tổng 2 điểm trước), `"D"` (gấp đôi điểm trước), `"C"` (hủy điểm trước). Trả tổng điểm.

In [28]:
def exercise_baseball(ops: list[str]) -> int:
    # TODO: dùng list làm stack
    pass

# assert exercise_baseball(["5", "2", "C", "D", "+"]) == 30
# assert exercise_baseball(["5", "-2", "4", "C", "D", "9", "+", "+"]) == 27

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Bắt đầu X = 0 | +1, −1, −1, −1, +1 | 1, 0, −1, −2, −1 |
| 2 | `op[0]` của `X--`, `X++` là `'X'` | cả hai bị coi là giảm | +1 −1 −1 |
| 3 | Khuôn mục 4 | `if op[1] == '+'` | `X += 1` / `X -= 1` |
| 4 | Đếm thao tác tăng | `sum(1 for op in ... if '+' in op)` | `2 * p - n` |
| 5 | `dict` + `get(name, 0)` | tên biến: `op[0] if op[0] not in '+-' else op[2]` | cộng ±1 theo `op[1]` |
| 6 | Stack lưu điểm hợp lệ | `"+"`: `stack[-1] + stack[-2]` | `"C"`: `pop()`, `"D"`: `2 * stack[-1]`, cuối `sum(stack)` |

## 18. Đáp án tham khảo, template và checklist

- Bài 1: 0 → **1 → 0 → −1 → −2 → −1**; kết quả **−1**.
- Bài 2: `wrong_first_char` coi `X--` và `X++` đều là giảm: +1 −1 −1 = **−1**; đáp án đúng: +1 −1 +1 = **1**.

In [29]:
def answer_final_value(operations):
    X = 0
    for op in operations:
        X += 1 if op[1] == '+' else -1
    return X

def answer_final_value_formula(operations):
    p = sum(1 for op in operations if '+' in op)
    return 2 * p - len(operations)

def answer_multi_vars(operations):
    values = {}
    for op in operations:
        name = op[0] if op[0] not in "+-" else op[2]
        values[name] = values.get(name, 0) + (1 if op[1] == '+' else -1)
    return values

def answer_baseball(ops):
    stack = []
    for op in ops:
        if op == "+":
            stack.append(stack[-1] + stack[-2])
        elif op == "D":
            stack.append(2 * stack[-1])
        elif op == "C":
            stack.pop()
        else:
            stack.append(int(op))
    return sum(stack)

assert answer_final_value(["++X", "X--", "X--", "--X", "X++"]) == -1
assert wrong_first_char(["++X", "X--", "X++"]) == -1 and answer_final_value(["++X", "X--", "X++"]) == 1
assert answer_final_value(["--X", "X++", "X++"]) == 1 and answer_final_value(["X++", "++X", "--X", "X--"]) == 0
assert answer_final_value_formula(["++X", "++X", "X++"]) == 3
assert answer_multi_vars(["++A", "B--", "A++", "--B", "C++"]) == {"A": 2, "B": -2, "C": 1}
assert answer_baseball(["5", "2", "C", "D", "+"]) == 30
assert answer_baseball(["5", "-2", "4", "C", "D", "9", "+", "+"]) == 27
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


### Template nhớ nhanh

```python
class Solution:
    def finalValueAfterOperations(self, operations: List[str]) -> int:
        X = 0
        for op in operations:
            X += 1 if op[1] == '+' else -1    # ký tự giữa luôn là dấu
        return X
```

Một dòng: `return sum(1 if op[1] == '+' else -1 for op in operations)`.

### Checklist trước khi submit
- [ ] Xử lý **cả** tiền tố (`++X`) **và** hậu tố (`X++`)?
- [ ] Kiểm tra ký tự **giữa** `op[1]` (không phải `op[0]`)?
- [ ] Dùng `X += 1` / `X -= 1` (Python không có `++`)?
- [ ] Khởi tạo `X = 0`?
- [ ] Đã test một thao tác mỗi loại và trường hợp X âm?